In [12]:
import numpy as np
import pandas as pd
import psycopg2
from tqdm import tqdm
from datetime import timedelta
from dateutil.relativedelta import relativedelta

In [13]:
sales = pd.read_csv('pedidos-1743765883268.csv', sep=';', encoding='latin1')

In [14]:
df = sales[['account_id', 'sales_channel_id']].value_counts()
df = pd.DataFrame(df).reset_index()

In [15]:
EPOCHS = 1000
BATCH_SIZE = 32
VALID_SPLIT = 0.1

In [16]:
LOOKBACK = 3
SEASONAL_PERIODS = (24, 24*7)
COVERAGE = 0.33
END_DATE = pd.to_datetime(pd.to_datetime(sales['created_date'].max()).strftime("%Y-%m-%d %H:00:00"))
START_DATE = END_DATE - timedelta(hours=END_DATE.hour)
TEST_SIZE = int((END_DATE - START_DATE).seconds / 60**2 + 1)

In [17]:
account_ids = ['05d9cc3a-decc-4fa2-b4e8-b44eef40e3b6', '4f1c8b04-eb4b-4887-8c69-a096f0fda3ab', 'ad942d1a-15ab-4a0d-83a8-ae82183ece53']

df_filtered = df[df['account_id'].isin(account_ids)]

In [18]:
# id_pairs = list(zip(df['account_id'], df['sales_channel_id']))
id_pairs = list(zip(df_filtered['account_id'], df_filtered['sales_channel_id']))

In [34]:
id_pairs

[('05d9cc3a-decc-4fa2-b4e8-b44eef40e3b6', 3),
 ('05d9cc3a-decc-4fa2-b4e8-b44eef40e3b6', 6),
 ('05d9cc3a-decc-4fa2-b4e8-b44eef40e3b6', 5),
 ('ad942d1a-15ab-4a0d-83a8-ae82183ece53', 1),
 ('ad942d1a-15ab-4a0d-83a8-ae82183ece53', 2),
 ('4f1c8b04-eb4b-4887-8c69-a096f0fda3ab', 1),
 ('05d9cc3a-decc-4fa2-b4e8-b44eef40e3b6', 1),
 ('4f1c8b04-eb4b-4887-8c69-a096f0fda3ab', 2),
 ('05d9cc3a-decc-4fa2-b4e8-b44eef40e3b6', 4),
 ('ad942d1a-15ab-4a0d-83a8-ae82183ece53', 3)]

In [27]:
weights_chan = {}
for account_id, sales_channel_id in tqdm(id_pairs):
    
    if sales_channel_id == 'ALL':
        cond = (sales['account_id'] == account_id) & \
               (sales['status'].notna())
    else:
        cond = (sales['account_id'] == account_id) & \
               (sales['sales_channel_id'] == sales_channel_id) & \
               (sales['status'].notna())

    df_client = sales[cond].drop(['account_id', 'sales_channel_id'], axis=1)
    df_client['created_date'] = pd.to_datetime(df_client['created_date'], format='%Y-%m-%d %H:%M:%S.%f %z')
    df_client = df_client.sort_values('created_date').reset_index(drop=True)
    df_client['created_date'] = df_client['created_date'].dt.strftime("%Y-%m-%d %H:00:00").reset_index(drop=True)

    df_client_mod = df_client.groupby('created_date').agg(price_total_agg=('price_total', 'sum'), n_orders=('created_date', 'count'))
    df_client_mod.index = pd.to_datetime(df_client_mod.index, format='%Y-%m-%d %H:00:00')

    end_date = pd.to_datetime(END_DATE, format='%Y-%m-%d %H:%M:%S')
    start_date = end_date - relativedelta(months=LOOKBACK)
    date_index = pd.Series(pd.date_range(start=start_date, end=end_date, freq='h', name='created_date'))
    df_client_mod = pd.merge(date_index, df_client_mod, how='left', on='created_date').set_index('created_date')
    
    weights_chan[account_id] = {} if account_id not in weights_chan else weights_chan[account_id]
    
    df = df_client_mod['n_orders'].fillna(0)
    df = df.loc[df.index <= END_DATE]
    
    weights_chan[account_id][sales_channel_id] = df.median()

100%|██████████| 10/10 [00:36<00:00,  3.64s/it]


In [28]:
weights_df = pd.DataFrame(weights_chan)

weights_df

,05d9cc3a-decc-4fa2-b4e8-b44eef40e3b6,ad942d1a-15ab-4a0d-83a8-ae82183ece53,4f1c8b04-eb4b-4887-8c69-a096f0fda3ab
3,68.0,0.0,NaN
6,42.0,NaN,NaN
5,6.0,NaN,NaN
1,0.0,2.0,8.0
4,0.0,NaN,NaN
2,NaN,3.0,3.0


In [29]:
weights_df = weights_df.stack().reset_index().rename(columns={'level_0': 'sales_channel_id', 'level_1': 'account_id', 0: 'weights'}).sort_values(by=['account_id', 'sales_channel_id'])

weights_df

,sales_channel_id,account_id,weights
4,1,05d9cc3a-decc-4fa2-b4e8-b44eef40e3b6,0.0
0,3,05d9cc3a-decc-4fa2-b4e8-b44eef40e3b6,68.0
7,4,05d9cc3a-decc-4fa2-b4e8-b44eef40e3b6,0.0
3,5,05d9cc3a-decc-4fa2-b4e8-b44eef40e3b6,6.0
2,6,05d9cc3a-decc-4fa2-b4e8-b44eef40e3b6,42.0
6,1,4f1c8b04-eb4b-4887-8c69-a096f0fda3ab,8.0
9,2,4f1c8b04-eb4b-4887-8c69-a096f0fda3ab,3.0
5,1,ad942d1a-15ab-4a0d-83a8-ae82183ece53,2.0
8,2,ad942d1a-15ab-4a0d-83a8-ae82183ece53,3.0
1,3,ad942d1a-15ab-4a0d-83a8-ae82183ece53,0.0


In [30]:
weights_chan_df = weights_df.groupby('account_id')['weights'].apply(lambda x: x / x.sum()).fillna(0).reset_index().rename(columns={0: 'weights'}).drop('level_1', axis=1)

weights_chan_df = pd.concat([weights_df['sales_channel_id'].reset_index(drop=True), weights_chan_df], axis=1)

weights_chan_df

,sales_channel_id,account_id,weights
0,1,05d9cc3a-decc-4fa2-b4e8-b44eef40e3b6,0.000000
1,3,05d9cc3a-decc-4fa2-b4e8-b44eef40e3b6,0.586207
2,4,05d9cc3a-decc-4fa2-b4e8-b44eef40e3b6,0.000000
3,5,05d9cc3a-decc-4fa2-b4e8-b44eef40e3b6,0.051724
4,6,05d9cc3a-decc-4fa2-b4e8-b44eef40e3b6,0.362069
5,1,4f1c8b04-eb4b-4887-8c69-a096f0fda3ab,0.727273
6,2,4f1c8b04-eb4b-4887-8c69-a096f0fda3ab,0.272727
7,1,ad942d1a-15ab-4a0d-83a8-ae82183ece53,0.400000
8,2,ad942d1a-15ab-4a0d-83a8-ae82183ece53,0.600000
9,3,ad942d1a-15ab-4a0d-83a8-ae82183ece53,0.000000


In [31]:
weights_acc_df = weights_df.groupby('account_id')['weights'].sum() / weights_df.groupby('account_id')['weights'].sum().sum()

weights_acc_df

account_id
05d9cc3a-decc-4fa2-b4e8-b44eef40e3b6    0.878788
4f1c8b04-eb4b-4887-8c69-a096f0fda3ab    0.083333
ad942d1a-15ab-4a0d-83a8-ae82183ece53    0.037879
Name: weights, dtype: float64

In [33]:
dbname = 'railway'
username = 'sinatra'
pwd = '781B3XjpeuqE'
hostname = 'monorail.proxy.rlwy.net'
port = 25096

connection = psycopg2.connect(database=dbname, user=username, password=pwd, host=hostname, port=port)
cursor = connection.cursor()

In [36]:
model = 'GradientBoosting'

In [43]:
df_forecast_metrics_norm = {}
scores = {}
df_forecast_metrics_norm[model] = {}

for account_id, sales_channel_id in id_pairs:
    
    print(account_id, sales_channel_id)
    
    # if sales_channel_id == 'ALL':
    #     cond = (sales['account_id'] == account_id) & \
    #            (sales['status'].notna())
    # else:
    #     cond = (sales['account_id'] == account_id) & \
    #            (sales['sales_channel_id'] == sales_channel_id) & \
    #            (sales['status'].notna())
    cond = (sales['account_id'] == account_id) & \
           (sales['sales_channel_id'] == sales_channel_id) & \
           (sales['status'].notna())

    df_client = sales[cond].drop(['account_id', 'sales_channel_id'], axis=1)
    df_client['created_date'] = pd.to_datetime(df_client['created_date'], format='%Y-%m-%d %H:%M:%S.%f %z')
    df_client = df_client.sort_values('created_date').reset_index(drop=True)
    df_client['created_date'] = df_client['created_date'].dt.strftime("%Y-%m-%d %H:00:00").reset_index(drop=True)

    df_client_mod = df_client.groupby('created_date').agg(price_total_agg=('price_total', 'sum'), n_orders=('created_date', 'count'))
    df_client_mod.index = pd.to_datetime(df_client_mod.index, format='%Y-%m-%d %H:00:00')

    end_date = pd.to_datetime(END_DATE, format='%Y-%m-%d %H:%M:%S')
    start_date = end_date - relativedelta(months=LOOKBACK)
    date_index = pd.Series(pd.date_range(start=start_date, end=end_date, freq='h', name='created_date'))
    df_client_mod = pd.merge(date_index, df_client_mod, how='left', on='created_date').set_index('created_date')
    
    df_forecast_metrics_norm[model][account_id] = {} if account_id not in df_forecast_metrics_norm[model] else df_forecast_metrics_norm[model][account_id]
    for dataset in ['sales', 'orders']:
        if dataset == 'sales':
            col = 'price_total_agg'
        else:
            col = 'n_orders'
        
        df = df_client_mod[col].fillna(0)
        
        y_test = df.loc[(df.index >= START_DATE) & (df.index <= END_DATE)].copy()
    
        cursor.execute(f"select {dataset}_high, {dataset}_low, {dataset}_mean from public.forecast where account_id = '{account_id}' and channel = '{sales_channel_id}' and model = '{model}'")
        res = cursor.fetchall()
        
        df_forecast = pd.DataFrame(res, columns=[f'{dataset}_high', f'{dataset}_low', f'{dataset}_mean'])[:9].set_index(y_test.index)
        
        display(df_forecast)
        
        forecast_mean = df_forecast[f'{dataset}_mean']
        forecast_low = df_forecast[f'{dataset}_low']
        forecast_high = df_forecast[f'{dataset}_high']
        
        forecast = pd.concat([y_test, forecast_mean, forecast_low, forecast_high], axis=1)
        forecast.columns = ['actual', 'forecast', 'lower', 'upper']
        
        display(forecast)
        
        df_forecast = forecast.assign(
            covered_pts=lambda x:
                4*x['actual'].between(x['lower'], x['upper'], inclusive='both') +
                2*(x['actual'].between(2*x['lower']-x['forecast'], x['lower'], inclusive='left') + x['actual'].between(x['upper'], 2*x['upper']-x['forecast'], inclusive='right')) +
                1*(x['actual'].between(3*x['lower']-2*x['forecast'], 2*x['lower']-x['forecast'], inclusive='left') + x['actual'].between(2*x['upper']-x['forecast'], 3*x['upper']-2*x['forecast'], inclusive='right')),
            covered_width=lambda x: x['upper'] - x['lower'],
        )
        
        display(df_forecast)
        
        df_forecast_metrics = {}
        df_forecast_metrics['total_covered'] = df_forecast['covered_pts'].sum()
        df_forecast_metrics['avg_covered'] = df_forecast['covered_pts'].mean()
        df_forecast_metrics['avg_covered_width'] = df_forecast['covered_width'].mean()
        
        display(df_forecast_metrics)
        
        if dataset == 'orders':
            df_forecast_orders_metrics_norm = df_forecast_metrics['avg_covered']/(1 + np.log(1 + df_forecast_metrics['avg_covered_width']))
            print('orders_metric:', df_forecast_orders_metrics_norm)
        else: 
            df_forecast_sales_metrics_norm = df_forecast_metrics['avg_covered']/(1 + np.log(1 + df_forecast_metrics['avg_covered_width']))
            print('sales_metric:', df_forecast_sales_metrics_norm)
    
    weight_chan = weights_chan_df.loc[(weights_chan_df['account_id'] == account_id) & (weights_chan_df['sales_channel_id'] == sales_channel_id), 'weights'].values[0]
    print(weight_chan)
    df_forecast_metrics_norm[model][account_id][sales_channel_id] = weight_chan * (0.5*df_forecast_sales_metrics_norm + 0.5*df_forecast_orders_metrics_norm)
    print(df_forecast_metrics_norm[model][account_id][sales_channel_id])

05d9cc3a-decc-4fa2-b4e8-b44eef40e3b6 3


,sales_high,sales_low,sales_mean
created_date,,,
2025-04-04 00:00:00,5270.203808,4808.226625,5100.822289
2025-04-04 01:00:00,3889.668714,3427.691531,3720.287195
2025-04-04 02:00:00,1715.070000,1294.026518,1545.688481
2025-04-04 03:00:00,851.464054,595.130000,639.642629
2025-04-04 04:00:00,959.067183,497.090000,745.173036
2025-04-04 05:00:00,642.470000,386.135946,430.648575
2025-04-04 06:00:00,1838.051519,1624.157371,1668.670000
2025-04-04 07:00:00,4872.306775,4615.972721,4660.485350
2025-04-04 08:00:00,12818.439542,12356.462359,12649.058023


,actual,forecast,lower,upper
created_date,,,,
2025-04-04 00:00:00,8263.33,5100.822289,4808.226625,5270.203808
2025-04-04 01:00:00,3262.63,3720.287195,3427.691531,3889.668714
2025-04-04 02:00:00,1715.07,1545.688481,1294.026518,1715.070000
2025-04-04 03:00:00,595.13,639.642629,595.130000,851.464054
2025-04-04 04:00:00,497.09,745.173036,497.090000,959.067183
2025-04-04 05:00:00,642.47,430.648575,386.135946,642.470000
2025-04-04 06:00:00,1668.67,1668.670000,1624.157371,1838.051519
2025-04-04 07:00:00,6766.89,4660.485350,4615.972721,4872.306775
2025-04-04 08:00:00,5432.42,12649.058023,12356.462359,12818.439542


,actual,forecast,lower,upper,covered_pts,covered_width
created_date,,,,,,
2025-04-04 00:00:00,8263.33,5100.822289,4808.226625,5270.203808,0,461.977183
2025-04-04 01:00:00,3262.63,3720.287195,3427.691531,3889.668714,2,461.977183
2025-04-04 02:00:00,1715.07,1545.688481,1294.026518,1715.070000,4,421.043482
2025-04-04 03:00:00,595.13,639.642629,595.130000,851.464054,2,256.334054
2025-04-04 04:00:00,497.09,745.173036,497.090000,959.067183,4,461.977183
2025-04-04 05:00:00,642.47,430.648575,386.135946,642.470000,2,256.334054
2025-04-04 06:00:00,1668.67,1668.670000,1624.157371,1838.051519,4,213.894148
2025-04-04 07:00:00,6766.89,4660.485350,4615.972721,4872.306775,0,256.334054
2025-04-04 08:00:00,5432.42,12649.058023,12356.462359,12818.439542,0,461.977183


{'total_covered': np.int64(18),
 'avg_covered': np.float64(2.0),
 'avg_covered_width': np.float64(361.3165026666667)}

...

,orders_high,orders_low,orders_mean
created_date,,,
2025-04-04 00:00:00,0.000000,0.000000,0.000000
2025-04-04 01:00:00,0.000000,0.000000,0.000000
2025-04-04 02:00:00,0.000000,0.000000,0.000000
2025-04-04 03:00:00,0.000000,0.000000,0.000000
2025-04-04 04:00:00,0.000000,0.000000,0.000000
2025-04-04 05:00:00,0.000000,0.000000,0.000000
2025-04-04 06:00:00,0.000000,0.000000,0.000000
2025-04-04 07:00:00,0.000000,0.000000,0.000000
2025-04-04 08:00:00,0.000000,0.000000,0.000000


,actual,forecast,lower,upper
created_date,,,,
2025-04-04 00:00:00,0.0,0.000000,0.000000,0.000000
2025-04-04 01:00:00,0.0,0.000000,0.000000,0.000000
2025-04-04 02:00:00,0.0,0.000000,0.000000,0.000000
2025-04-04 03:00:00,0.0,0.000000,0.000000,0.000000
2025-04-04 04:00:00,0.0,0.000000,0.000000,0.000000
2025-04-04 05:00:00,0.0,0.000000,0.000000,0.000000
2025-04-04 06:00:00,0.0,0.000000,0.000000,0.000000
2025-04-04 07:00:00,0.0,0.000000,0.000000,0.000000
2025-04-04 08:00:00,0.0,0.000000,0.000000,0.000000


,actual,forecast,lower,upper,covered_pts,covered_width
created_date,,,,,,
2025-04-04 00:00:00,0.0,0.000000,0.000000,0.000000,4,0.000000
2025-04-04 01:00:00,0.0,0.000000,0.000000,0.000000,4,0.000000
2025-04-04 02:00:00,0.0,0.000000,0.000000,0.000000,4,0.000000
2025-04-04 03:00:00,0.0,0.000000,0.000000,0.000000,4,0.000000
2025-04-04 04:00:00,0.0,0.000000,0.000000,0.000000,4,0.000000
2025-04-04 05:00:00,0.0,0.000000,0.000000,0.000000,4,0.000000
2025-04-04 06:00:00,0.0,0.000000,0.000000,0.000000,4,0.000000
2025-04-04 07:00:00,0.0,0.000000,0.000000,0.000000,4,0.000000
2025-04-04 08:00:00,0.0,0.000000,0.000000,0.000000,4,0.000000


{'total_covered': np.int64(36),
 'avg_covered': np.float64(4.0),
 'avg_covered_width': np.float64(0.0)}

orders_metric: 4.0
0.0
0.0


In [47]:
0.5862068965517241*(0.5*0.9405500038466145 + 0.5*0.2901697111945687)

0.36072819233965714

In [44]:
scores[model] = np.sum([weights_acc_df[acc_id] * np.sum(list(df_forecast_metrics_norm[model][acc_id].values())) for acc_id in df_forecast_metrics_norm[model].keys()])

In [45]:
scores

{'GradientBoosting': np.float64(0.5922541434192474)}

In [46]:
df_forecast_metrics_norm

{'GradientBoosting': {'05d9cc3a-decc-4fa2-b4e8-b44eef40e3b6': {3: np.float64(0.36072819233965714),
   6: np.float64(0.16924621616107324),
   5: np.float64(0.04076184212530444),
   1: np.float64(0.0),
   4: np.float64(0.0)},
  'ad942d1a-15ab-4a0d-83a8-ae82183ece53': {1: np.float64(0.36236033167517345),
   2: np.float64(0.5143257184334704),
   3: np.float64(0.0)},
  '4f1c8b04-eb4b-4887-8c69-a096f0fda3ab': {1: np.float64(0.4614813414916191),
   2: np.float64(0.22840153197905375)}}}